# Experiment 04: Easy Condition Recovery

Main easy-condition recovery experiment for linear and tanh models.

In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path | None = None) -> Path:
    current = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src" / "config.py").exists() and (candidate / "README.md").exists():
            return candidate
    raise RuntimeError("Could not locate repository root.")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repo root: {REPO_ROOT}")


In [ ]:
import numpy as np
import pandas as pd

from src import ExperimentConfig, run_experiment


def overall_row(results: dict) -> pd.Series:
    summary_df = results["summary_df"]
    return summary_df.loc[summary_df["set_idx"] == "overall"].iloc[0]


def collect_overall_rows(result_map: dict[str, dict], columns: list[str] | None = None) -> pd.DataFrame:
    rows = []
    for label, results in result_map.items():
        row = overall_row(results).copy()
        row["run_label"] = label
        rows.append(row)
    df = pd.DataFrame(rows)
    if columns is None:
        return df
    keep = [col for col in ["run_label", *columns] if col in df.columns]
    return df[keep]


In [ ]:
BASE_CONFIG = dict(
    time_mode="discrete",
    SEQ_LEN=4096,
    theta_min=0.05 * np.pi,
    theta_max=0.85 * np.pi,
    MIN_DELTA_THETA=0.12 * np.pi,
    RANDOM_AMPLITUDE=False,
    RANDOM_PHASE=True,
    USE_NOISE=False,
    HIDDEN_DIM=128,
    BOTTLENECK_MULTIPLIER=4,
    NUM_EXPERIMENTS=20,
    SEEDS_PER_FREQ=5,
    VAL_RATIO=0.2,
    TEST_RATIO=0.2,
    NORMALIZE_H_COLUMNS=False,
    VERBOSE=False,
    MAKE_PLOTS=False,
)


In [ ]:
MODEL_CONFIGS = {
    "linear": dict(MODEL_ID="AN003_LINEAR", LR=1e-2),
    "tanh": dict(MODEL_ID="AN002_NO_BN_TANH", LR=1e-3),
}
FIXED_ARGS = dict(NUM_FREQS=4, LAG=32, BOTTLENECK_MULTIPLIER=4, EPOCHS=1500)


In [ ]:
results = {}
for model_name, model_kwargs in MODEL_CONFIGS.items():
    cfg = ExperimentConfig(**BASE_CONFIG, **FIXED_ARGS, **model_kwargs)
    results[model_name] = run_experiment(cfg)

collect_overall_rows(
    results,
    columns=[
        "train_mse_mean",
        "test_mse_mean",
        "test_align_coverage_full_mean",
        "test_recon_r2_qf_from_h_mean",
        "test_align_mean_angle_deg_full_mean",
    ],
)
